# Rightmove Property Address Scraper

This notebook scrapes property addresses from Rightmove search results.

## Instructions:
1. Run each cell in order from top to bottom
2. When prompted, upload your Chrome extension (.zip or .crx file)
3. Upload your outcodes JSON file
4. The scraper will automatically process all outcodes and save results to Excel

## What you'll need:
- Chrome extension file for authentication/session management
- JSON file with outcodes in format: `[{"code":1,"outcode":"AB10"},{"code":2,"outcode":"AB11"}, ...]`

## Step 1: Install Required Packages
This cell installs all necessary libraries. Wait for it to complete before moving to the next step.

In [ ]:
!apt-get update
!apt-get install -y chromium-chromedriver
!pip install selenium openpyxl pandas

import sys
sys.path.insert(0, '/usr/lib/chromium-browser/chromedriver')

print("✅ All packages installed successfully!")

## Step 2: Import Libraries

In [ ]:
import json
import time
import os
import zipfile
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from google.colab import files

print("✅ Libraries imported successfully!")

## Step 3: Upload Chrome Extension
Click the 'Choose Files' button and select your Chrome extension file (.zip or .crx)

In [ ]:
print("📤 Please upload your Chrome extension file...")
uploaded_extension = files.upload()

# Get the uploaded filename
extension_filename = list(uploaded_extension.keys())[0]
extension_path = os.path.abspath(extension_filename)

print(f"✅ Extension uploaded: {extension_filename}")
print(f"📁 Saved to: {extension_path}")

## Step 4: Upload Outcodes JSON File
Upload your JSON file containing the outcodes to search

In [ ]:
print("📤 Please upload your outcodes JSON file...")
uploaded_json = files.upload()

# Get the uploaded filename and load the data
json_filename = list(uploaded_json.keys())[0]
with open(json_filename, 'r') as f:
    outcodes_data = json.load(f)

print(f"✅ Outcodes loaded: {len(outcodes_data)} outcodes found")
print(f"📋 Preview: {outcodes_data[:3]}...")

## Step 5: Initialize Browser with Extension
This sets up Chrome with your extension

In [ ]:
def setup_driver(extension_path):
    """Initialize Chrome driver with extension loaded"""
    chrome_options = Options()
    chrome_options.add_argument('--headless')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--disable-gpu')
    chrome_options.add_argument('--window-size=1920,1080')
    
    # Load the extension
    if extension_path.endswith('.zip'):
        # If it's a zip file, extract it first
        extract_path = '/tmp/extension'
        os.makedirs(extract_path, exist_ok=True)
        with zipfile.ZipFile(extension_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        chrome_options.add_argument(f'--load-extension={extract_path}')
    else:
        # Assume it's a .crx file or unpacked extension
        chrome_options.add_extension(extension_path)
    
    service = Service('/usr/bin/chromedriver')
    driver = webdriver.Chrome(service=service, options=chrome_options)
    return driver

print("✅ Browser setup function ready!")

## Step 6: Define Scraping Functions

In [ ]:
def build_rightmove_url(outcode, index=0):
    """Build Rightmove search URL for given outcode and page index"""
    base_url = "https://www.rightmove.co.uk/property-for-sale/find.html"
    # Format: OUTCODE%5E{outcode}
    location_identifier = f"OUTCODE%5E{outcode}"
    url = f"{base_url}?locationIdentifier={location_identifier}&index={index}"
    return url

def extract_addresses_from_page(driver):
    """Extract property addresses from current page"""
    addresses = []
    
    try:
        # Wait for the page to load
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_all_located((By.TAG_NAME, "a"))
        )
        
        # Find all property cards/listings
        # Look for links that contain property addresses
        property_links = driver.find_elements(By.CSS_SELECTOR, "a[target='_blank'][rel='noopener noreferrer']")
        
        for link in property_links:
            try:
                # Get the address text from the link
                address_text = link.text.strip()
                
                if address_text:
                    # Check if this is a "only postcode found" entry
                    # Look for the parent or nearby elements
                    parent = link.find_element(By.XPATH, "./ancestor::*[1]")
                    parent_html = parent.get_attribute('innerHTML')
                    
                    # Check if "Only postcode found" text is present
                    if '📍 Only postcode found' in parent_html or 'Only postcode found' in parent_html:
                        print(f"  ⏭️  Skipping (only postcode): {address_text[:50]}...")
                        continue
                    
                    # Valid address found
                    addresses.append(address_text)
                    print(f"  ✓ Found: {address_text[:50]}...")
            except Exception as e:
                # Skip problematic elements
                continue
    
    except TimeoutException:
        print("  ⚠️  Page load timeout")
    except Exception as e:
        print(f"  ⚠️  Error extracting addresses: {str(e)}")
    
    return addresses

def has_next_page(driver):
    """Check if there's a next page available"""
    try:
        # Look for next page button or pagination indicator
        next_buttons = driver.find_elements(By.CSS_SELECTOR, "button[data-test='pagination-next']")
        if next_buttons and next_buttons[0].is_enabled():
            return True
        
        # Alternative: check for next page link
        next_links = driver.find_elements(By.LINK_TEXT, "Next")
        if next_links:
            return True
            
        return False
    except:
        return False

def scrape_outcode(driver, outcode):
    """Scrape all pages for a given outcode"""
    print(f"\n🔍 Scraping outcode: {outcode}")
    all_addresses = []
    page_num = 1
    index = 0
    
    while True:
        print(f"  📄 Page {page_num} (index={index})")
        url = build_rightmove_url(outcode, index)
        
        try:
            driver.get(url)
            time.sleep(2)  # Wait for page to load
            
            # Extract addresses from current page
            addresses = extract_addresses_from_page(driver)
            
            if not addresses:
                print(f"  ℹ️  No addresses found on page {page_num}. End of results.")
                break
            
            all_addresses.extend(addresses)
            print(f"  📊 Found {len(addresses)} addresses on this page")
            
            # Check if there's a next page
            if not has_next_page(driver):
                print(f"  ℹ️  No more pages available")
                break
            
            # Move to next page (increment index by 24)
            index += 24
            page_num += 1
            
            # Safety limit to avoid infinite loops
            if page_num > 100:
                print(f"  ⚠️  Reached page limit (100 pages)")
                break
                
        except Exception as e:
            print(f"  ❌ Error on page {page_num}: {str(e)}")
            break
    
    print(f"✅ Completed {outcode}: {len(all_addresses)} total addresses found")
    return all_addresses

print("✅ Scraping functions defined!")

## Step 7: Run the Scraper
This will process all outcodes and collect addresses. This may take several minutes depending on the number of outcodes.

In [ ]:
# Initialize results storage
all_results = []

# Setup the driver
print("🚀 Initializing browser...")
driver = setup_driver(extension_path)

try:
    # Process each outcode
    total_outcodes = len(outcodes_data)
    print(f"\n📋 Processing {total_outcodes} outcodes...\n")
    
    for idx, outcode_entry in enumerate(outcodes_data, 1):
        outcode = outcode_entry.get('outcode', '')
        
        if not outcode:
            print(f"⚠️  Skipping entry {idx}: no outcode found")
            continue
        
        print(f"\n{'='*60}")
        print(f"Progress: {idx}/{total_outcodes} outcodes")
        
        # Scrape addresses for this outcode
        addresses = scrape_outcode(driver, outcode)
        
        # Store results with outcode information
        for address in addresses:
            all_results.append({
                'Outcode': outcode,
                'Address': address
            })
        
        # Small delay between outcodes
        time.sleep(1)
    
    print(f"\n{'='*60}")
    print(f"\n🎉 Scraping completed!")
    print(f"📊 Total addresses collected: {len(all_results)}")
    
finally:
    # Always close the driver
    driver.quit()
    print("\n✅ Browser closed")

## Step 8: Export to Excel
Save all collected addresses to an Excel file and download it

In [ ]:
if all_results:
    # Create DataFrame
    df = pd.DataFrame(all_results)
    
    # Generate filename with timestamp
    from datetime import datetime
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_filename = f'rightmove_addresses_{timestamp}.xlsx'
    
    # Save to Excel
    df.to_excel(output_filename, index=False, engine='openpyxl')
    
    print(f"✅ Excel file created: {output_filename}")
    print(f"📊 Total rows: {len(df)}")
    print(f"\n📋 Preview of results:")
    print(df.head(10))
    
    # Download the file
    print(f"\n⬇️  Downloading file...")
    files.download(output_filename)
    print(f"✅ Download started! Check your downloads folder.")
else:
    print("⚠️  No addresses were collected. Nothing to export.")

## Summary

✅ **Done!** Your property addresses have been scraped and exported to Excel.

### What happened:
1. Chrome browser was set up with your extension
2. Each outcode was searched on Rightmove
3. All pages were scraped for each outcode (incrementing by 24)
4. Addresses were extracted from property links
5. Entries with "Only postcode found" were filtered out
6. Results were saved to an Excel file

### Troubleshooting:
- If you got few or no results, the extension might need manual authentication
- Try adjusting the wait times in the scraping functions
- Check that your outcodes JSON format matches the expected structure

### Need to run again?
Simply re-run all cells from the top. You'll be prompted to re-upload files.